# Lecture 7 — Class Exercise
## Heatmap & Waterfall: Netflix Catalogue

## Task 1 — Heatmap: content by rating and release decade

**What to build:** A heatmap showing the number of titles by **content rating** (y-axis) and **decade** (x-axis).

**Requirements:**
- Create a 'decade' column: `df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'`
- Filter to TV-14, TV-MA, PG-13, R, PG only (most common ratings)
- Sequential colour scale (Blues)
- Values shown in cells (`text_auto=True`)
- Insight title about which rating dominates which decade

---


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('../../data/netflix_catalogue.csv')
print(f"Loaded: {len(df)} titles")
print(df['type'].value_counts())
print(df.head())

print("Genres:", df['genre'].value_counts().head(8))
print("\nCountries:", df['country'].value_counts().head(8))
print("\nRatings:", df['rating'].value_counts())

df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'

target_ratings = ['TV-14', 'TV-MA', 'PG-13', 'R', 'PG']
df_filtered = df[df['rating'].isin(target_ratings)].copy()

heatmap_data = df_filtered.groupby(['rating', 'decade']).size().reset_index(name='count')
pivot_matrix = heatmap_data.pivot(index='rating', columns='decade', values='count').fillna(0)

pivot_matrix = pivot_matrix.reindex(columns=sorted(pivot_matrix.columns))

fig = px.imshow(
    pivot_matrix,
    labels=dict(x="Release Decade", y="Content Rating", color="Number of Titles"),
    x=pivot_matrix.columns,
    y=pivot_matrix.index,
    color_continuous_scale='Blues',       
    text_auto=True,                        
    title="TV-MA and TV-14 Rated Content Dominates Netflix Catalogue Distributions in the 2010s" 
)

fig.update_layout(
    coloraxis_showscale=True,
    margin=dict(t=60, l=40, r=40, b=40)
)

fig.show()

Loaded: 3000 titles
type
Movie      1974
TV Show    1026
Name: count, dtype: int64
      type  release_year  added_year             genre        country rating  \
0    Movie          2014        2016  Sci-Fi & Fantasy         France  PG-13   
1    Movie          2010        2014     Documentaries  United States  TV-MA   
2  TV Show          2011        2012     Kids & Family  United States  TV-14   
3    Movie          2016        2018             Anime          India     PG   
4    Movie          2014        2016     Kids & Family         Canada  TV-MA   

   duration  
0       157  
1       127  
2         6  
3       134  
4        77  
Genres: genre
Sports                244
Sci-Fi & Fantasy      213
Kids & Family         209
Crime                 206
Drama                 204
Horror                199
Action & Adventure    198
Thrillers             195
Name: count, dtype: int64

Countries: country
United States     932
India             337
United Kingdom    261
Japan             

## Task 2 — Waterfall: Movie vs TV Show additions by year

**What to build:** A waterfall chart showing how Netflix's **Movie library** grew year by year (2015-2022).

**Requirements:**
- Filter to Movies only
- Group by `added_year`, count titles per year
- Final bar should be the cumulative total
- Green bars (additions), blue total
- Annotation on the year with the largest single addition
- Insight title naming the growth story


In [ ]:
import pandas as pd
import plotly.graph_objects as go

df_movies = df[
    (df["type"] == "Movie") & (df["added_year"] >= 2015) & (df["added_year"] <= 2022)
].copy()

yearly_counts = (
    df_movies.groupby("added_year").size().reset_index(name="count")
)

years = yearly_counts["added_year"].astype(str).tolist()
counts = yearly_counts["count"].tolist()

max_idx = yearly_counts["count"].idxmax()
max_year = years[max_idx]
max_value = counts[max_idx]

years.append("Total Library")
counts.append(sum(counts))

measure = ["relative"] * len(yearly_counts) + ["total"]

# 3. Build the Waterfall Chart
fig = go.Figure(
    go.Waterfall(
        name="Movie Additions",
        orientation="v",
        measure=measure,
        x=years,
        y=counts,
        textposition="outside",
        text=[f"+{c}" if m == "relative" else f"{c}" for m, c in zip(measure, counts)],
        decreasing=dict(marker=dict(color="red")),  
        increasing=dict(marker=dict(color="#2ca02c")),  
        totals=dict(marker=dict(color="#1f77b4")),  
    )
)

fig.add_annotation(
    x=max_year,
    y=max_value,
    text=f"<b>Peak Growth Year</b><br>Netflix added a record<br>{max_value} movies in {max_year}.",
    showarrow=True,
    arrowhead=2,
    ax=0,
    ay=-50,  
    bgcolor="white",
    bordercolor="black",
    borderwidth=1,
)

fig.update_layout(
    title="Netflix Rapidly Expanded Its Movie Collection, Reaching Peak Growth in 2019 Before Slowing Down",  # Requirement: Insight title
    xaxis_title="Year Added to Netflix",
    yaxis_title="Number of Movies",
    waterfallgap=0.3,
)

fig.show()